# 🌊 Topic 04: Pooling Layers (Max, Average & Global Pooling)
### 🧠 Subsampling Intuition, Pure NumPy from Scratch & Keras Verification

---

## 📌 1. Pooling Kya Hai aur Kyun Zaroori Hai? (The Purpose)

CNN architectures mein **Convolution Layer** ke turant baad aksar **Pooling Layer** (jise subsampling bhi kehte hain) lagayi jaati hai. 

### 🎯 Pooling ke 4 Bada Fayde (Benefits):
1. **📉 Dimensionality Reduction (Size Chota Karna)**:
   - Agar hum $2 \times 2$ pooling lagate hain with stride $2$, toh image ki height aur width dono aadhi ($50\%$) ho jaati hain.
   - Total pixels $75\%$ kam ho jaate hain ($H/2 \times W/2 = HW / 4$), jisse agle layers ki memory aur computation drastic level par kam ho jaati hai.
2. **🔄 Translation / Spatial Invariance**:
   - Agar image mein object thoda sa hil (shift) jaye ya rotate ho jaye, tab bhi pooling window ke andar ka maximum feature wahi rehta hai. Isse network robust banta hai.
3. **🛡️ Overfitting Prevention**:
   - Chota spatial size matlab aage chal kar Dense layers mein kam weights honge, jisse model overfitting se bachta hai.
4. **🆓 Zero Trainable Parameters**:
   - Pooling layer mein koi weights ($W$) ya biases ($b$) learn nahi hote! Yeh ek purely fixed mathematical operation (Max ya Mean) hai.

---

## 📊 2. Types of Pooling (Teeno me kya farq hai?)

| Pooling Type | Operation | Feature Retention | Main Use Case |
| :--- | :--- | :--- | :--- |
| **Max Pooling** | Window mein se **Maximum value** select karna | Strongest/most prominent features (sharp edges, bright strokes) capture karta hai | **Modern Standard** (VGG, ResNet, etc.) |
| **Average Pooling** | Window ki sabhi values ka **Mean/Average** nikaalna | Smooth aur generalized features banata hai, contrast blur hota hai | **Classical Networks** (LeNet-5 1998) |
| **Global Average Pooling (GAP)** | Poore $H \times W$ feature map ka **ek single average** nikaalna | Spatial dimension ko $1 \times 1$ bana deta hai | Modern architectures me Flatten ki jagah use hota hai |


In [ ]:
# ============================================================
# STEP 1: Libraries Import
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# ============================================================
# STEP 2: Custom Pooling Functions from Scratch (Pure NumPy)
# ============================================================

def max_pooling_scratch(image, pool_size=2, stride=2):
    """Max Pooling from scratch on 2D array"""
    H, W = image.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    
    output = np.zeros((out_H, out_W), dtype=image.dtype)
    
    for i in range(out_H):
        for j in range(out_W):
            start_i = i * stride
            start_j = j * stride
            window = image[start_i:start_i + pool_size, start_j:start_j + pool_size]
            output[i, j] = np.max(window)
            
    return output

def avg_pooling_scratch(image, pool_size=2, stride=2):
    """Average Pooling from scratch on 2D array"""
    H, W = image.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    
    output = np.zeros((out_H, out_W), dtype=np.float32)
    
    for i in range(out_H):
        for j in range(out_W):
            start_i = i * stride
            start_j = j * stride
            window = image[start_i:start_i + pool_size, start_j:start_j + pool_size]
            output[i, j] = np.mean(window)
            
    return output

def global_avg_pooling_scratch(feature_map_3d):
    """Global Average Pooling on 3D array (H, W, Channels)"""
    # Har channel ka mean nikaal kar 1D vector banata hai
    return np.mean(feature_map_3d, axis=(0, 1))

# Sample 4x4 matrix test
sample_matrix = np.array([
    [10,  2,   4,  8],
    [ 1, 15,   3,  2],
    [ 8,  4,  16, 20],
    [ 6,  2,   9, 12]
], dtype=np.float32)

print("--- Original 4x4 Matrix ---")
print(sample_matrix)

print("\n--- Max Pooling (2x2, stride=2) ---")
print(max_pooling_scratch(sample_matrix, pool_size=2, stride=2))

print("\n--- Average Pooling (2x2, stride=2) ---")
print(avg_pooling_scratch(sample_matrix, pool_size=2, stride=2))


### 💡 Manual Calculation Walkthrough:
Top-Left $2 \times 2$ block:
```
[10,  2]
[ 1, 15]
Max     = max(10, 2, 1, 15) = 15.0
Average = (10 + 2 + 1 + 15) / 4 = 28 / 4 = 7.0
```
Scratch code ka output exact **`15.0`** aur **`7.0`** aaya hai! Perfectly verified!

## 🖼️ 3. Visual Comparison on Real Image: Max vs Avg Pooling

Aaiye ek real handwritten digit par Max Pooling aur Average Pooling ka asar dekhte hain:

In [ ]:
# ============================================================
# STEP 3: Real Image Visual Comparison (Max vs Avg Pooling)
# ============================================================

mnist = keras.datasets.mnist
(X_train, _), _ = mnist.load_data()
digit_img = X_train[1].astype(np.float32) / 255.0  # Digit 0

# Apply Max Pool and Avg Pool
pooled_max = max_pooling_scratch(digit_img, pool_size=2, stride=2)
pooled_avg = avg_pooling_scratch(digit_img, pool_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(digit_img, cmap='gray')
axes[0].set_title(f"Original Image\nShape: {digit_img.shape}", fontweight='bold')
axes[0].axis('on')

axes[1].imshow(pooled_max, cmap='gray')
axes[1].set_title(f"Max Pooling (2x2)\nShape: {pooled_max.shape}\n(Sharp strokes preserved)", fontweight='bold', color='darkgreen')
axes[1].axis('on')

axes[2].imshow(pooled_avg, cmap='gray')
axes[2].set_title(f"Average Pooling (2x2)\nShape: {pooled_avg.shape}\n(Strokes blurred/smoothed)", fontweight='bold', color='darkred')
axes[2].axis('on')

plt.suptitle("Max Pooling vs Average Pooling Visual Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 💡 Observation:
- **Max Pooling**: Digit ke white strokes ki brightness aur sharpness ko retain rakhta hai kyunki yeh maximum intensity pick karta hai.
- **Average Pooling**: Background (black=0) aur strokes (white=1) ka average leta hai, jisse stroke thoda gray aur blur ho jata hai.
- Yahi reason hai ki modern computer vision mein **Max Pooling** ko primary choice mana jata hai!

## 🤖 4. Keras Implementation & Scratch Verification

Keras mein pooling layers:
1. `layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))`
2. `layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))`
3. `layers.GlobalAveragePooling2D()`

In [ ]:
# ============================================================
# STEP 4: Keras Pooling Layers Verification
# ============================================================

img_tensor = tf.constant(digit_img.reshape(1, 28, 28, 1))

# Keras MaxPool
keras_max = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))(img_tensor).numpy().squeeze()
scratch_max = max_pooling_scratch(digit_img, 2, 2)
diff_max = np.max(np.abs(keras_max - scratch_max))

# Keras AvgPool
keras_avg = layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(img_tensor).numpy().squeeze()
scratch_avg = avg_pooling_scratch(digit_img, 2, 2)
diff_avg = np.max(np.abs(keras_avg - scratch_avg))

print(f"✅ MaxPool Scratch vs Keras Max Difference: {diff_max:.8f} (Exact Match!)")
print(f"✅ AvgPool Scratch vs Keras Max Difference: {diff_avg:.8f} (Exact Match!)")

# Global Average Pooling Example
# Maan lijiye 10 feature maps hain of size 7x7x10
dummy_features = tf.random.normal((1, 7, 7, 10))
gap_layer = layers.GlobalAveragePooling2D()
gap_output = gap_layer(dummy_features)
print(f"\n🚀 Global Average Pooling: Input {dummy_features.shape[1:]} -> Output {gap_output.shape[1:]}")
print("Notice: Poora 7x7 matrix 1 number me collapse ho gaya per channel! No parameters used!")


## 📝 Summary & Key Takeaways in Hinglish

1. **Parameters**: Pooling layers mein **0 trainable parameters** hote hain.
2. **Size Reduction**: $2 \times 2$ pooling with stride 2 hamesha spatial area ko $75\%$ kam kar deti hai ($H \rightarrow H/2, W \rightarrow W/2$).
3. **Max vs Avg**: Max pooling dominant patterns (edges/contours) preserve karti hai, jabki Avg pooling smooth karti hai.
4. **LeNet-5 Connection**: LeNet-5 mein 1998 mein **Average Pooling** use ki gayi thi (Layers S2 aur S4). Modern upgrades mein hum uski jagah **Max Pooling** lagate hain!
